In [24]:
import os
import json
import random
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')   # non-interactive backend (no display needed)
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision import models, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# ── local modules ──────────────────────────────────────────────────────────────
from dataset import StanfordDogs, load_synset_mapping
from generate_bbox import get_bboxes_from_heatmap
from metric import compute_localization_metrics

 CONFIGURATION

In [25]:
ROOT           = '.\\data\\SDOGS_SMALL'           # Stanford Dogs root directory
WEIGHTS_DIR    = ".\\weights\\"    
MAP_PATH       = 'imagenet_class_index.json'
IMAGE_SIZE     = 224
BATCH_SIZE     = 16
NUM_WORKERS    = 0                        # set >0 if multiprocessing works
SEED           = 42
OUTPUT_DIR     = 'outputs'
VIZ_MODEL      = 'resnet18'              # model used for visualisation
VIZ_N_SAMPLES  = 6                       # how many random images to visualise

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)

# ── model weight paths ─────────────────────────────────────────────────────────
MODEL_CONFIGS = {
    'resnet18':   (models.resnet18,      WEIGHTS_DIR + 'resnet18_weights.pth'),
    'resnet101':  (models.resnet101,     WEIGHTS_DIR + 'resnet101-63fe2227.pth'),
    'squeezenet': (models.squeezenet1_1, WEIGHTS_DIR + 'squeezenet1_1-b8a52dc0.pth'),
    'convnext':   (models.convnext_tiny, WEIGHTS_DIR + 'convnext_tiny-983f1562.pth'),
    'googlenet':  (models.googlenet,     WEIGHTS_DIR + 'googlenet-1378be20.pth'),
}

DATASET

In [26]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

synset_map = load_synset_mapping(MAP_PATH)

dataset = StanfordDogs(
    root=ROOT,
    synset_to_imagenet_idx=synset_map,
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print(f"Dataset size: {len(dataset)} images")

Loaded 1200 samples.
Dataset size: 1200 images


CAM EXTRACTOR

In [35]:
class CAMExtractor:
    """
    Extracts a Class Activation Map for a given image & class index.

    The CAM formula (Zhou et al. 2016, Eq. 2):
        M_c(x,y) = Σ_k  w_k^c · f_k(x, y)

    where:
        f_k(x,y)  = activation of feature-map k at spatial position (x,y)
                    from the LAST convolutional layer
        w_k^c     = weight connecting feature-map k to class c in the
                    classification head (after GAP)

    Each architecture exposes these differently; we handle each case below.
    """

    def __init__(self, model, model_name: str):
        self.model      = model
        self.model_name = model_name
        self.feature_maps = None   # will be filled by the forward hook
        self._register_hook()

    # ── hook registration ──────────────────────────────────────────────────────
    def _register_hook(self):
        """Attach a forward hook to the last convolutional layer."""
        layer = self._get_last_conv_layer()
        layer.register_forward_hook(self._hook_fn)

    def _hook_fn(self, module, input, output):
        """Store the feature maps (B, C, H, W) produced by the last conv."""
        # detach so we don't accidentally backprop through it
        self.feature_maps = output.detach()

    def _get_last_conv_layer(self):
        """Return the last convolutional layer for each architecture."""
        name = self.model_name
        if name in ('resnet18', 'resnet101'):
            # ResNets: last layer is model.layer4[-1].conv2
            # But we want the output of the entire layer4 block (after BN+ReLU)
            # → hook onto the last BasicBlock / Bottleneck
            return list(self.model.layer4.children())[-1]

        elif name == 'squeezenet':
            # SqueezeNet1_1: features is a Sequential; last conv is index 12
            # (a Fire module's expand3x3 conv)
            return self.model.features[-1]   # last Fire module

        elif name == 'convnext':
            # ConvNeXt-Tiny: model.features is Sequential of stages;
            # last stage is index 7
            return self.model.features[-1]

        elif name == 'googlenet':
            # GoogLeNet: last inception block before the avgpool
            return self.model.inception5b

        else:
            raise ValueError(f"Unknown model: {name}")

    # ── weight extraction ──────────────────────────────────────────────────────
    def _get_cam_weights(self, class_idx: int) -> torch.Tensor:
        """
        Return the weight vector w^c of shape (num_feature_maps,).

        For GAP-based classifiers:  weights come from the final FC / Linear layer.
        For ConvNeXt:               the classifier is Sequential[LayerNorm, Flatten, Linear].
        For SqueezeNet:             the classifier contains a Conv2d(512,1000,1x1).
        """
        name = self.model_name

        if name in ('resnet18', 'resnet101'):
            # model.fc: Linear(512 or 2048 → 1000)
            weights = self.model.fc.weight           # (1000, C)
            return weights[class_idx]                # (C,)

        elif name == 'squeezenet':
            # model.classifier[1]: Conv2d(512, 1000, kernel_size=1)
            # weights shape: (1000, 512, 1, 1) → squeeze to (1000, 512)
            conv_w = self.model.classifier[1].weight  # (1000, 512, 1, 1)
            return conv_w[class_idx].squeeze()        # (512,)

        elif name == 'convnext':
            # model.classifier: Sequential(LayerNorm, Flatten, Linear(768,1000))
            linear = self.model.classifier[2]         # Linear
            return linear.weight[class_idx]           # (768,)

        elif name == 'googlenet':
            # model.fc: Linear(1024 → 1000)
            return self.model.fc.weight[class_idx]    # (1024,)

        else:
            raise ValueError(f"Unknown model: {name}")

    # ── main CAM computation ───────────────────────────────────────────────────
    def get_cam(self, image_tensor: torch.Tensor, class_idx: int) -> np.ndarray:
        """
        Run a forward pass and compute the CAM for `class_idx`.

        Steps:
          1. Forward pass  →  hook fills self.feature_maps  (1, C, h, w)
          2. Get weights   →  w  shape (C,)
          3. Weighted sum  →  cam = Σ_k w_k * f_k           shape (h, w)
          4. ReLU          →  keep only positive activations
          5. Normalise     →  scale to [0, 255] uint8
          6. Upsample      →  resize to IMAGE_SIZE × IMAGE_SIZE

        Returns:
            cam_img: np.ndarray uint8, shape (IMAGE_SIZE, IMAGE_SIZE)
        """
        self.model.eval()
        with torch.no_grad():
            _ = self.model(image_tensor.unsqueeze(0))   # triggers hook

        # feature_maps: (1, C, h, w)
        fmaps = self.feature_maps[0]        # (C, h, w)
        weights = self._get_cam_weights(class_idx)  # (C,)

        # weighted sum:  cam(x,y) = Σ_k  w_k · f_k(x,y)
        # einsum: 'k, k h w -> h w'
        cam = torch.einsum('k, k h w -> h w', weights.cpu(), fmaps.cpu())

        # ReLU — suppress negative values (they don't contribute positively
        # to the class score and just add noise)
        cam = F.relu(cam)

        # Normalise to [0, 255]
        cam_np = cam.detach().numpy()
        if cam_np.max() > cam_np.min():
            cam_np = (cam_np - cam_np.min()) / (cam_np.max() - cam_np.min())
        cam_np = (cam_np * 255).astype(np.uint8)

        # Upsample to input resolution
        cam_np = cv2.resize(cam_np, (IMAGE_SIZE, IMAGE_SIZE),
                            interpolation=cv2.INTER_LINEAR)
        return cam_np


MODEL LOADER

In [28]:
def load_model(model_name: str) -> nn.Module:
    """
    Instantiate the architecture and load pretrained weights from disk.
    Returns the model in eval mode on CPU.
    """
    constructor, weights_path = MODEL_CONFIGS[model_name]
    model = constructor(pretrained=False)

    state = torch.load(weights_path, map_location='cpu')
    # some checkpoints store a nested dict
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']

    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        print(f"  [warn] missing keys:    {missing[:3]} ...")
    if unexpected:
        print(f"  [warn] unexpected keys: {unexpected[:3]} ...")

    model.eval()
    return model

EVALUATION — Classification + Localization

In [29]:
def scale_bbox_to_original(bbox_224, orig_w, orig_h):
    """
    The dataset resizes images to 224×224 before feeding the model,
    so the bounding boxes from GT are in original pixel coords while
    the predicted bboxes live in 224×224 space.

    We scale the predicted bbox back to original image coordinates
    so IoU is computed consistently.

    bbox_224 format: [x, y, w, h]  (output of get_bboxes_from_heatmap)
    Returns: [x1, y1, x2, y2] in original image coords
    """
    x, y, w, h = bbox_224
    scale_x = orig_w / IMAGE_SIZE
    scale_y = orig_h / IMAGE_SIZE
    x1 = x  * scale_x
    y1 = y  * scale_y
    x2 = (x + w) * scale_x
    y2 = (y + h) * scale_y
    return [x1, y1, x2, y2]


def evaluate_model(model_name: str):
    """
    Full evaluation pipeline for one model:
      - Top-1 / Top-5 classification accuracy
      - CAM generation → bbox extraction → IoU metrics

    Returns a dict with all metrics.
    """
    print(f"\n{'='*60}")
    print(f"  Evaluating: {model_name}")
    print(f"{'='*60}")

    model = load_model(model_name)
    extractor = CAMExtractor(model, model_name)

    top1_correct = 0
    top5_correct = 0
    total        = 0

    gt_bboxes   = []
    pred_bboxes = []

    for batch in tqdm(loader, desc=f"{model_name}"):
        images, bboxes_gt, class_indices, orig_sizes = batch
        # images:       (B, 3, 224, 224)
        # bboxes_gt:    (B, 4)  format: [xmin, ymin, xmax, ymax]
        # class_indices:(B,)    ImageNet class index (int)
        # orig_sizes:   list of [w, h]

        B = images.size(0)
        total += B

        # ── Classification ────────────────────────────────────────────────
        with torch.no_grad():
            logits = model(images)          # (B, 1000)

        # Top-1
        pred_top1 = logits.argmax(dim=1)   # (B,)
        top1_correct += (pred_top1 == class_indices).sum().item()

        # Top-5
        top5_preds = logits.topk(5, dim=1).indices   # (B, 5)
        for i in range(B):
            if class_indices[i].item() in top5_preds[i].tolist():
                top5_correct += 1

        # ── Localization ──────────────────────────────────────────────────
        for i in range(B):
            # Use predicted class for CAM (as in the paper's top-1 setting)
            pred_class = pred_top1[i].item()

            cam_img = extractor.get_cam(images[i], pred_class)

            # get_bboxes_from_heatmap returns [x, y, w, h]
            try:
                bbox_pred_xywh = get_bboxes_from_heatmap(cam_img)
            except Exception:
                # fallback: entire image
                bbox_pred_xywh = [0, 0, IMAGE_SIZE, IMAGE_SIZE]

            orig_w, orig_h = orig_sizes[0][i].item(), orig_sizes[1][i].item()

            # Scale predicted bbox to original image coordinates
            bbox_pred_xyxy = scale_bbox_to_original(
                bbox_pred_xywh, orig_w, orig_h
            )

            gt_bboxes.append(bboxes_gt[i].tolist())    # [x1,y1,x2,y2]
            pred_bboxes.append(bbox_pred_xyxy)

    # ── Compute metrics ───────────────────────────────────────────────────────
    top1_acc = top1_correct / total * 100
    top5_acc = top5_correct / total * 100

    gt_tensor   = torch.tensor(gt_bboxes,   dtype=torch.float32)
    pred_tensor = torch.tensor(pred_bboxes, dtype=torch.float32)

    loc_metrics = compute_localization_metrics(
        gt_tensor, pred_tensor, iou_thresholds=[0.5, 0.75]
    )

    results = {
        'model':         model_name,
        'top1_acc':      top1_acc,
        'top5_acc':      top5_acc,
        'mean_iou':      loc_metrics['mean_iou'],
        'median_iou':    loc_metrics['median_iou'],
        'iou@0.5':       loc_metrics['accuracy@0.5'],
        'iou@0.75':      loc_metrics['accuracy@0.75'],
    }

    print(f"\n  Top-1 Accuracy : {top1_acc:.2f}%")
    print(f"  Top-5 Accuracy : {top5_acc:.2f}%")
    print(f"  Mean IoU       : {loc_metrics['mean_iou']:.4f}")
    print(f"  Median IoU     : {loc_metrics['median_iou']:.4f}")
    print(f"  IoU @ 0.50     : {loc_metrics['accuracy@0.5']:.2f}%")
    print(f"  IoU @ 0.75     : {loc_metrics['accuracy@0.75']:.2f}%")

    return results

VISUALISATION — Heatmap & Bounding Box

In [30]:
def denormalize(tensor):
    """
    Undo ImageNet normalisation to get a displayable RGB image.
    tensor: (3, H, W) normalised tensor
    Returns: (H, W, 3) uint8 numpy array
    """
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img  = tensor * std + mean
    img  = img.clamp(0, 1).permute(1, 2, 0).numpy()
    return (img * 255).astype(np.uint8)


def overlay_cam(image_rgb, cam_img, alpha=0.45):
    """
    Blend the CAM heatmap over the original image.

    image_rgb: (H, W, 3) uint8
    cam_img:   (H, W)   uint8  [0..255]
    Returns:   (H, W, 3) uint8 blended image
    """
    heatmap = cv2.applyColorMap(cam_img, cv2.COLORMAP_JET)   # BGR
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)        # RGB
    blended = cv2.addWeighted(image_rgb, 1 - alpha, heatmap, alpha, 0)
    return blended


def visualize_heatmaps_and_boxes(model_name: str, n_samples: int = 6):
    """
    For `n_samples` randomly chosen images:
      - Row 1: original image with GT bounding box (green)
      - Row 2: CAM heatmap overlaid on image
      - Row 3: image with predicted bounding box (red) and GT box (green)

    Saves the figure to OUTPUT_DIR.
    """
    print(f"\n  Generating visualisations for: {model_name}")

    model     = load_model(model_name)
    extractor = CAMExtractor(model, model_name)

    # Pick random samples
    indices = random.sample(range(len(dataset)), n_samples)

    fig, axes = plt.subplots(3, n_samples, figsize=(4 * n_samples, 12))
    fig.suptitle(
        f'CAM Visualisation — {model_name.upper()}',
        fontsize=16, fontweight='bold', y=1.01
    )

    row_labels = ['Original + GT Box', 'CAM Heatmap', 'Predicted Box (red) + GT (green)']
    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=11, rotation=90, labelpad=8)

    for col, idx in enumerate(indices):
        image_t, bbox_gt, class_idx, orig_size = dataset[idx]
        orig_w, orig_h = orig_size

        # ── Get CAM ──────────────────────────────────────────────────────
        with torch.no_grad():
            logits = model(image_t.unsqueeze(0))
        pred_class = logits.argmax(dim=1).item()

        cam_img = extractor.get_cam(image_t, pred_class)

        # ── Predicted bbox (in 224 space, then scale up) ──────────────
        try:
            bbox_pred_xywh = get_bboxes_from_heatmap(cam_img)
        except Exception:
            bbox_pred_xywh = [0, 0, IMAGE_SIZE, IMAGE_SIZE]

        pred_xyxy = scale_bbox_to_original(bbox_pred_xywh, orig_w, orig_h)
        gt_xyxy   = bbox_gt.tolist()   # [x1, y1, x2, y2] in original coords

        # ── Decode image for display ─────────────────────────────────────
        img_rgb = denormalize(image_t)

        # Scale img to original size for bbox drawing
        img_orig = cv2.resize(img_rgb, (orig_w, orig_h))

        # ── Row 0: original + GT box ─────────────────────────────────────
        ax = axes[0, col]
        ax.imshow(img_orig)
        x1g, y1g, x2g, y2g = gt_xyxy
        rect = patches.Rectangle(
            (x1g, y1g), x2g - x1g, y2g - y1g,
            linewidth=2, edgecolor='lime', facecolor='none'
        )
        ax.add_patch(rect)
        ax.set_title(f'cls {class_idx}', fontsize=9)
        ax.axis('off')

        # ── Row 1: CAM heatmap overlay ───────────────────────────────────
        ax = axes[1, col]
        cam_overlay = overlay_cam(img_rgb, cam_img)   # in 224 space
        ax.imshow(cam_overlay)
        ax.axis('off')

        # ── Row 2: predicted + GT boxes ──────────────────────────────────
        ax = axes[2, col]
        ax.imshow(img_orig)
        # GT (green)
        rect_gt = patches.Rectangle(
            (x1g, y1g), x2g - x1g, y2g - y1g,
            linewidth=2, edgecolor='lime', facecolor='none', label='GT'
        )
        ax.add_patch(rect_gt)
        # Predicted (red)
        x1p, y1p, x2p, y2p = pred_xyxy
        rect_pr = patches.Rectangle(
            (x1p, y1p), x2p - x1p, y2p - y1p,
            linewidth=2, edgecolor='red', facecolor='none', label='Pred'
        )
        ax.add_patch(rect_pr)
        ax.axis('off')

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, f'cam_visualization_{model_name}.png')
    plt.savefig(save_path, bbox_inches='tight', dpi=120)
    plt.close()
    print(f"  Saved → {save_path}")

RESULTS TABLE

In [31]:
def print_results_table(all_results: list):
    header = (
        f"{'Model':<12} {'Top-1':>8} {'Top-5':>8} "
        f"{'mIoU':>8} {'medIoU':>8} {'IoU@.5':>8} {'IoU@.75':>9}"
    )
    sep = '-' * len(header)
    print(f"\n{sep}")
    print(header)
    print(sep)
    for r in all_results:
        print(
            f"{r['model']:<12} "
            f"{r['top1_acc']:>7.2f}% "
            f"{r['top5_acc']:>7.2f}% "
            f"{r['mean_iou']:>8.4f} "
            f"{r['median_iou']:>8.4f} "
            f"{r['iou@0.5']:>7.2f}% "
            f"{r['iou@0.75']:>8.2f}%"
        )
    print(sep)

    # save as JSON for the report
    json_path = os.path.join(OUTPUT_DIR, 'results.json')
    with open(json_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f"\nResults saved → {json_path}")

MAIN

In [36]:
if __name__ == '__main__':

    all_results = []

    # ── A) Evaluate every model ───────────────────────────────────────────────
    for model_name in MODEL_CONFIGS:
        results = evaluate_model(model_name)
        all_results.append(results)

    print_results_table(all_results)

    # ── B) Visualise one model ────────────────────────────────────────────────
    visualize_heatmaps_and_boxes(VIZ_MODEL, n_samples=VIZ_N_SAMPLES)

    print("\nAll done!")

f:\vu\ANN\project2\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
f:\vu\ANN\project2\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



  Evaluating: resnet18


resnet18: 100%|██████████| 75/75 [01:52<00:00,  1.50s/it]



  Top-1 Accuracy : 81.75%
  Top-5 Accuracy : 97.25%
  Mean IoU       : 0.5206
  Median IoU     : 0.5125
  IoU @ 0.50     : 52.08%
  IoU @ 0.75     : 22.08%

  Evaluating: resnet101


resnet101: 100%|██████████| 75/75 [08:30<00:00,  6.81s/it]


...


  Top-1 Accuracy : 67.25%
  Top-5 Accuracy : 91.00%
  Mean IoU       : 0.5146
  Median IoU     : 0.5055
  IoU @ 0.50     : 51.00%
  IoU @ 0.75     : 19.92%

  Evaluating: convnext


convnext: 100%|██████████| 75/75 [04:29<00:00,  3.59s/it]
f:\vu\ANN\project2\.venv\Lib\site-packages\torchvision\models\googlenet.py:47: FutureWarning: The default weight initialization of GoogleNet will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(



  Top-1 Accuracy : 94.67%
  Top-5 Accuracy : 99.67%
  Mean IoU       : 0.5114
  Median IoU     : 0.5051
  IoU @ 0.50     : 50.58%
  IoU @ 0.75     : 21.25%

  Evaluating: googlenet


googlenet: 100%|██████████| 75/75 [03:57<00:00,  3.17s/it]



  Top-1 Accuracy : 70.58%
  Top-5 Accuracy : 92.75%
  Mean IoU       : 0.5257
  Median IoU     : 0.5135
  IoU @ 0.50     : 52.42%
  IoU @ 0.75     : 22.17%

-------------------------------------------------------------------
Model           Top-1    Top-5     mIoU   medIoU   IoU@.5   IoU@.75
-------------------------------------------------------------------
resnet18       81.75%   97.25%   0.5206   0.5125   52.08%    22.08%
resnet101      90.67%   99.08%   0.5137   0.5079   50.92%    21.17%
squeezenet     67.25%   91.00%   0.5146   0.5055   51.00%    19.92%
convnext       94.67%   99.67%   0.5114   0.5051   50.58%    21.25%
googlenet      70.58%   92.75%   0.5257   0.5135   52.42%    22.17%
-------------------------------------------------------------------

Results saved → outputs\results.json

  Generating visualisations for: resnet18
  Saved → outputs\cam_visualization_resnet18.png

All done!
